In [1]:
import time

import whisperx

start_time = time.time()
device = "cpu" # "cuda"
audio_file = "media_452785_msg_452785.oga"
batch_size = 16 # reduce if low on GPU mem
compute_type = "int8" # "float16" # change to "int8" if low on GPU mem (may reduce accuracy)

In [2]:
import os
import torch
from typing import Optional, Union
from pyannote.audio import Pipeline
from whisperx.diarize import DiarizationPipeline
from whisperx.log_utils import get_logger

logger = get_logger(__name__)

/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [3]:
model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root="./models")

/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2025-12-13 20:40:36 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2025-12-13 20:40:36 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint .venv/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.4.0. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.8.0+cu128. Bad things might happen unless you revert torch to 1.x.


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [4]:
transcribasiotn_start_time = time.time()
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
print(result["segments"]) # before alignment

2025-12-13 20:40:36 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2025-12-13 20:40:44 - whisperx.asr - INFO - Detected language: ru (0.95) in first 30s of audio
[{'text': ' Привет, как дела, что делаешь?', 'start': 0.875, 'end': 2.596}]


In [5]:
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

In [6]:
print(result["segments"]) # after alignment

print(f" Took time: {time.time() - start_time} seconds")
print(f" Transcribastion took time: {time.time() - transcribasiotn_start_time} seconds")

[{'start': 0.875, 'end': 2.616, 'text': ' Привет, как дела, что делаешь?', 'words': [{'word': 'Привет,', 'start': np.float64(0.875), 'end': np.float64(1.305), 'score': np.float64(0.809)}, {'word': 'как', 'start': np.float64(1.346), 'end': np.float64(1.49), 'score': np.float64(0.903)}, {'word': 'дела,', 'start': np.float64(1.551), 'end': np.float64(1.838), 'score': np.float64(1.0)}, {'word': 'что', 'start': np.float64(1.879), 'end': np.float64(2.043), 'score': np.float64(0.981)}, {'word': 'делаешь?', 'start': np.float64(2.084), 'end': np.float64(2.616), 'score': np.float64(0.968)}]}]
 Took time: 36.4218590259552 seconds
 Transcribastion took time: 26.580469846725464 seconds


In [7]:

class LocalDiarizationPipeline(DiarizationPipeline):
    """
    DiarizationPipeline с гарантированной загрузкой модели из локального диска
    (через HuggingFace cache).
    """

    def __init__(
        self,
        model_name: str = "pyannote/speaker-diarization-3.1",
        device: Optional[Union[str, torch.device]] = "cpu",
        cache_dir: Optional[str] = None,
        use_auth_token: Optional[str] = None,
        offline: bool = True,
    ):
        if isinstance(device, str):
            device = torch.device(device)

        # Жёстко включаем оффлайн-режим (по желанию)
        if offline:
            os.environ.setdefault("HF_HUB_OFFLINE", "1")
            os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

        logger.info(
            f"Loading diarization model (local): {model_name}, cache_dir={cache_dir}"
        )

        # ПЕРЕОПРЕДЕЛЯЕМ self.model вместо вызова super().__init__()
        self.model = Pipeline.from_pretrained(
            model_name,
            cache_dir=cache_dir,
            use_auth_token=use_auth_token,
        ).to(device)

In [8]:
diarize_model = LocalDiarizationPipeline(
    model_name="pyannote/speaker-diarization-3.1",
    device=device,  # cuda или "cpu"
    cache_dir="/home/slava/Documents/projects/Hanzo/wisper_python31013/models",
    offline=True,
)

2025-12-13 20:41:05 - whisperx - INFO - Loading diarization model (local): pyannote/speaker-diarization-3.1, cache_dir=/home/slava/Documents/projects/Hanzo/wisper_python31013/models


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [9]:
diarize_segments = diarize_model(audio, min_speakers=2)

/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_diarization.py:554: UserWarning: 
The detected number of speakers (1) is outside
the given bounds [2, inf]. This can happen if the
given audio file is too short to contain 2 or more speakers.
Try to lower the desired minimal number of speakers.

  warnings.warn(


In [10]:
result = whisperx.assign_word_speakers(diarize_segments, result)

In [11]:
print("diarize_segments")
print(diarize_segments)
print("result")
print(result["segments"]) # segments are now assigned speaker IDs

diarize_segments
                             segment label     speaker     start       end  \
0  [ 00:00:00.773 -->  00:00:02.730]     A  SPEAKER_00  0.773469  2.730969   

   intersection   union  
0         0.532  1.9575  
result
[{'start': 0.875, 'end': 2.616, 'text': ' Привет, как дела, что делаешь?', 'words': [{'word': 'Привет,', 'start': np.float64(0.875), 'end': np.float64(1.305), 'score': np.float64(0.809), 'speaker': 'SPEAKER_00'}, {'word': 'как', 'start': np.float64(1.346), 'end': np.float64(1.49), 'score': np.float64(0.903), 'speaker': 'SPEAKER_00'}, {'word': 'дела,', 'start': np.float64(1.551), 'end': np.float64(1.838), 'score': np.float64(1.0), 'speaker': 'SPEAKER_00'}, {'word': 'что', 'start': np.float64(1.879), 'end': np.float64(2.043), 'score': np.float64(0.981), 'speaker': 'SPEAKER_00'}, {'word': 'делаешь?', 'start': np.float64(2.084), 'end': np.float64(2.616), 'score': np.float64(0.968), 'speaker': 'SPEAKER_00'}], 'speaker': 'SPEAKER_00'}]


In [12]:
for s in result.get("segments"):
    #print(s)

    print(f"'{s.get('start')}-{s.get('end')}' : {s.get('speaker')} : {s.get('text')} ")

'0.875-2.616' : SPEAKER_00 :  Привет, как дела, что делаешь? 


In [13]:
import os
os.environ

environ{'INVOCATION_ID': '1afd0a08de584d119939ccb2a953fdec',
        'USERNAME': 'slava',
        'GSM_SKIP_SSH_AGENT_WORKAROUND': 'true',
        'SHLVL': '0',
        'PYTHONUNBUFFERED': '1',
        'XDG_DATA_DIRS': '/usr/share/ubuntu-xorg:/usr/share/gnome:/usr/local/share/:/usr/share/:/var/lib/snapd/desktop',
        'SHELL': '/bin/bash',
        'GJS_DEBUG_TOPICS': 'JS ERROR;JS LOG',
        'WINDOWPATH': '2',
        'PYTHONIOENCODING': 'UTF-8',
        'GJS_DEBUG_OUTPUT': 'stderr',
        'GNOME_SHELL_SESSION_MODE': 'ubuntu',
        'SESSION_MANAGER': 'local/office:@/tmp/.ICE-unix/4133,unix/office:/tmp/.ICE-unix/4133',
        'XDG_SESSION_CLASS': 'user',
        'GTK_MODULES': 'gail:atk-bridge',
        'DISPLAY': ':0',
        'HOME': '/home/slava',
        'MEMORY_PRESSURE_WATCH': '/sys/fs/cgroup/user.slice/user-1000.slice/user@1000.service/session.slice/org.gnome.Shell@x11.service/memory.pressure',
        'XDG_CURRENT_DESKTOP': 'ubuntu:GNOME',
        'PATH': '/home/slava